# Phase 6: AI-Generated Face Detection Extension (StyleGAN Dataset)

**Objective:** Fine-tune the existing DFDC-trained ViT model on a **mixed dataset** (DFDC + StyleGAN) to build a dual-capability deepfake detector that can identify both face-swap deepfakes and AI-generated faces.

**Critical Design Decision — Mixed Training:**
Training on StyleGAN data alone would cause catastrophic forgetting of the DFDC face-swap features. This notebook uses `ConcatDataset` to merge both datasets, ensuring the model continuously sees both data distributions during every epoch.

**Strategy:**
1. **Step 6.1:** Data audit & compatibility verification
2. **Step 6.2:** Mixed dataset setup with `ConcatDataset`
3. **Step 6.3:** Load DFDC checkpoint (`best_vit_deepfake_2.pth`) + freeze backbone
4. **Step 6.4:** Configure hyperparameters for fine-tuning
5. **Step 6.5:** Train with backbone freeze/unfreeze + evaluate per-dataset

# Step 6.1: Data Audit & Compatibility Verification

Verify both datasets exist, check image counts/dimensions, confirm label mapping compatibility, and spot-check for corruption.

In [1]:
import os
import random
from PIL import Image

# Dataset paths
DFDC_ROOT = "processed_faces_v2"
STYLEGAN_ROOT = "Fake_face_detection_with_Keras/real_vs_fake/real-vs-fake"

print("=" * 60)
print("  STEP 6.1: Data Audit & Compatibility Verification")
print("=" * 60)

# --- 1. Verify folder structures & count images ---
print("\n📁 Folder Structure & Image Counts:\n")

datasets_info = {
    "DFDC": {
        "train/FAKE": os.path.join(DFDC_ROOT, "train", "FAKE"),
        "train/REAL": os.path.join(DFDC_ROOT, "train", "REAL"),
        "val/FAKE":   os.path.join(DFDC_ROOT, "val", "FAKE"),
        "val/REAL":   os.path.join(DFDC_ROOT, "val", "REAL"),
    },
    "StyleGAN": {
        "train/fake": os.path.join(STYLEGAN_ROOT, "train", "fake"),
        "train/real": os.path.join(STYLEGAN_ROOT, "train", "real"),
        "valid/fake": os.path.join(STYLEGAN_ROOT, "valid", "fake"),
        "valid/real": os.path.join(STYLEGAN_ROOT, "valid", "real"),
        "test/fake":  os.path.join(STYLEGAN_ROOT, "test", "fake"),
        "test/real":  os.path.join(STYLEGAN_ROOT, "test", "real"),
    },
}

for ds_name, splits in datasets_info.items():
    print(f"  {ds_name} Dataset:")
    for split_label, path in splits.items():
        exists = os.path.isdir(path)
        count = len([f for f in os.listdir(path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if exists else 0
        status = "✅" if exists else "❌"
        print(f"    {status} {split_label}: {count:,} images")
    print()

# --- 2. Image dimension check ---
print("📐 Image Dimension Samples:")
sample_dirs = [
    ("DFDC",     os.path.join(DFDC_ROOT, "train", "FAKE")),
    ("StyleGAN", os.path.join(STYLEGAN_ROOT, "train", "fake")),
]
for ds_name, folder in sample_dirs:
    files = sorted([f for f in os.listdir(folder)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:3]
    for f in files:
        img = Image.open(os.path.join(folder, f))
        print(f"  {ds_name}: {f} → {img.size} ({img.mode})")
print()

# --- 3. Label mapping compatibility ---
print("🏷️  Label Mapping Compatibility:")
print("  DFDC     → ImageFolder class_to_idx: {'FAKE': 0, 'REAL': 1}")
print("  StyleGAN → ImageFolder class_to_idx: {'fake': 0, 'real': 1}")
print("  ✅ Numeric indices are identical (fake=0, real=1).")
print("     ConcatDataset can be used directly without label remapping.\n")

# --- 4. Corruption spot-check ---
print("🔍 Corruption Spot-Check (50 random samples per split):")
corrupted_count = 0
total_checked = 0

for ds_name, splits in datasets_info.items():
    for split_label, path in splits.items():
        if not os.path.isdir(path):
            continue
        files = [f for f in os.listdir(path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        sample = random.sample(files, min(50, len(files)))
        for f in sample:
            total_checked += 1
            try:
                img = Image.open(os.path.join(path, f))
                img.verify()
            except Exception as e:
                corrupted_count += 1
                print(f"  ❌ Corrupted: {path}/{f} — {e}")

if corrupted_count == 0:
    print(f"  ✅ All {total_checked} sampled images passed integrity check.")
else:
    print(f"  ⚠️  {corrupted_count}/{total_checked} corrupted files detected.")

print("\n" + "=" * 60)
print("  Data Audit Complete — Ready to proceed to Step 6.2")
print("=" * 60)

  STEP 6.1: Data Audit & Compatibility Verification

📁 Folder Structure & Image Counts:

  DFDC Dataset:
    ✅ train/FAKE: 18,953 images
    ✅ train/REAL: 18,959 images
    ✅ val/FAKE: 4,748 images
    ✅ val/REAL: 4,919 images

  StyleGAN Dataset:
    ✅ train/fake: 50,000 images
    ✅ train/real: 50,000 images
    ✅ valid/fake: 10,000 images
    ✅ valid/real: 10,000 images
    ✅ test/fake: 10,000 images
    ✅ test/real: 10,000 images

📐 Image Dimension Samples:
  DFDC: aabdnomlru_f000000.jpg → (332, 452) (RGB)
  DFDC: aabdnomlru_f000021.jpg → (319, 443) (RGB)
  DFDC: aabdnomlru_f000042.jpg → (319, 446) (RGB)
  StyleGAN: 001DDU0NI4.jpg → (256, 256) (RGB)
  StyleGAN: 002KDWZBHU.jpg → (256, 256) (RGB)
  StyleGAN: 002PMM0QG9.jpg → (256, 256) (RGB)

🏷️  Label Mapping Compatibility:
  DFDC     → ImageFolder class_to_idx: {'FAKE': 0, 'REAL': 1}
  StyleGAN → ImageFolder class_to_idx: {'fake': 0, 'real': 1}
  ✅ Numeric indices are identical (fake=0, real=1).
     ConcatDataset can be used direc

# Step 6.2: Mixed-Dataset Fine-tuning Setup

Load both datasets, subsample StyleGAN to match DFDC size, and merge with `ConcatDataset`.

In [2]:
import os
import torch
import numpy as np
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, ConcatDataset, Subset
from transformers import ViTImageProcessor

# ============================================================
# Device Setup — explicit MPS for Apple M2
# ============================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ============================================================
# Dataset Paths
# ============================================================
DFDC_ROOT = "processed_faces_v2"
STYLEGAN_ROOT = "Fake_face_detection_with_Keras/real_vs_fake/real-vs-fake"

# ============================================================
# ViT Processor Config & Transforms (matching Phase 2)
# ============================================================
model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_name)
mean = processor.image_mean
std  = processor.image_std
size = processor.size.get('height', 224)
print(f"Image config: {size}x{size}, mean={mean}, std={std}")

train_transforms = transforms.Compose([
    transforms.Resize((size, size)),           # Downscale 256→224 for StyleGAN
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_transforms = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# ============================================================
# Load Individual Datasets
# ============================================================
dfdc_train     = ImageFolder(root=os.path.join(DFDC_ROOT, "train"),         transform=train_transforms)
dfdc_val       = ImageFolder(root=os.path.join(DFDC_ROOT, "val"),           transform=val_transforms)

stylegan_train = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "train"),     transform=train_transforms)
stylegan_val   = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "valid"),     transform=val_transforms)
stylegan_test  = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "test"),      transform=val_transforms)

print(f"\nRaw dataset sizes:")
print(f"  DFDC train:     {len(dfdc_train):>7,} images  class_to_idx: {dfdc_train.class_to_idx}")
print(f"  DFDC val:       {len(dfdc_val):>7,} images  class_to_idx: {dfdc_val.class_to_idx}")
print(f"  StyleGAN train: {len(stylegan_train):>7,} images  class_to_idx: {stylegan_train.class_to_idx}")
print(f"  StyleGAN val:   {len(stylegan_val):>7,} images  class_to_idx: {stylegan_val.class_to_idx}")
print(f"  StyleGAN test:  {len(stylegan_test):>7,} images  class_to_idx: {stylegan_test.class_to_idx}")

# Verify label compatibility
assert list(dfdc_train.class_to_idx.values()) == [0, 1], "DFDC label mapping unexpected!"
assert list(stylegan_train.class_to_idx.values()) == [0, 1], "StyleGAN label mapping unexpected!"
print("\n✅ Label mapping verified: both datasets use fake=0, real=1")

# ============================================================
# Subsample StyleGAN to match DFDC size
# ============================================================
target_size = len(dfdc_train)  # ~37,912
rng = np.random.RandomState(42)
stylegan_indices = rng.choice(len(stylegan_train), size=target_size, replace=False)
stylegan_train_subset = Subset(stylegan_train, stylegan_indices)
print(f"\nStyleGAN train subsampled: {len(stylegan_train):,} → {len(stylegan_train_subset):,}")

# ============================================================
# Create Mixed Datasets
# ============================================================
combined_train = ConcatDataset([dfdc_train, stylegan_train_subset])
combined_val   = ConcatDataset([dfdc_val, stylegan_val])

print(f"\nCombined training set:   {len(combined_train):,} images "
      f"(DFDC: {len(dfdc_train):,} + StyleGAN: {len(stylegan_train_subset):,})")
print(f"Combined validation set: {len(combined_val):,} images "
      f"(DFDC: {len(dfdc_val):,} + StyleGAN: {len(stylegan_val):,})")

# ============================================================
# DataLoaders
# ============================================================
BATCH_SIZE   = 8
NUM_WORKERS  = 2

train_loader = DataLoader(
    combined_train, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=False
)
combined_val_loader = DataLoader(
    combined_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

# Separate val/test loaders for per-dataset evaluation
dfdc_val_loader = DataLoader(
    dfdc_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)
stylegan_val_loader = DataLoader(
    stylegan_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)
stylegan_test_loader = DataLoader(
    stylegan_test, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

print(f"\nDataLoaders created:")
print(f"  train_loader (mixed):        {len(train_loader):,} batches")
print(f"  combined_val_loader (mixed):  {len(combined_val_loader):,} batches")
print(f"  dfdc_val_loader:              {len(dfdc_val_loader):,} batches")
print(f"  stylegan_val_loader:          {len(stylegan_val_loader):,} batches")
print(f"  stylegan_test_loader:         {len(stylegan_test_loader):,} batches")

Using device: mps


Image config: 224x224, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)

Raw dataset sizes:
  DFDC train:      37,912 images  class_to_idx: {'FAKE': 0, 'REAL': 1}
  DFDC val:         9,667 images  class_to_idx: {'FAKE': 0, 'REAL': 1}
  StyleGAN train: 100,000 images  class_to_idx: {'fake': 0, 'real': 1}
  StyleGAN val:    20,000 images  class_to_idx: {'fake': 0, 'real': 1}
  StyleGAN test:   20,000 images  class_to_idx: {'fake': 0, 'real': 1}

✅ Label mapping verified: both datasets use fake=0, real=1

StyleGAN train subsampled: 100,000 → 37,912

Combined training set:   75,824 images (DFDC: 37,912 + StyleGAN: 37,912)
Combined validation set: 29,667 images (DFDC: 9,667 + StyleGAN: 20,000)

DataLoaders created:
  train_loader (mixed):        9,478 batches
  combined_val_loader (mixed):  3,709 batches
  dfdc_val_loader:              1,209 batches
  stylegan_val_loader:          2,500 batches
  stylegan_test_loader:         2,500 batches


# Step 6.3: Transfer Learning — Load Checkpoint & Freeze Backbone

Load `best_vit_deepfake_2.pth` (trained on DFDC, ROC-AUC 0.9738) and freeze the ViT backbone.
Only the classification head is trainable for the first 2 epochs to prevent destabilizing gradients.

In [3]:
import torch.nn as nn
from transformers import ViTModel

# ============================================================
# Model Architecture (identical to Phase 2)
# ============================================================
class ViTDeepfakeClassifier(nn.Module):
    def __init__(self, vit_model, dropout_rate=0.2):
        super().__init__()
        self.vit = vit_model
        self.dropout = nn.Dropout(dropout_rate)
        # Classification head: single binary logit output
        self.classifier = nn.Linear(self.vit.config.hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        # Extract [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

# ============================================================
# Load Base ViT + DFDC Checkpoint
# ============================================================
print(f"Loading base ViT from: {model_name}")
vit_base = ViTModel.from_pretrained(model_name)

model = ViTDeepfakeClassifier(vit_base, dropout_rate=0.2)

checkpoint_path = "best_vit_deepfake_2.pth"
print(f"Loading DFDC checkpoint: {checkpoint_path}")
state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
print("✅ Checkpoint loaded successfully.")

# ============================================================
# Freeze ViT Backbone
# ============================================================
model.vit.requires_grad_(False)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"\nParameter Summary (Backbone FROZEN):")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}  (classification head only)")
print(f"  Frozen parameters:    {frozen_params:>12,}  (ViT backbone)")
print(f"  Trainable ratio:      {trainable_params / total_params * 100:.2f}%")

model = model.to(device)
print(f"\nModel placed on device: {device}")

Loading base ViT from: google/vit-base-patch16-224-in21k


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading DFDC checkpoint: best_vit_deepfake_2.pth
✅ Checkpoint loaded successfully.

Parameter Summary (Backbone FROZEN):
  Total parameters:       86,390,017
  Trainable parameters:          769  (classification head only)
  Frozen parameters:      86,389,248  (ViT backbone)
  Trainable ratio:      0.00%

Model placed on device: mps


# Step 6.4: Hyperparameter Configuration for Fine-tuning

Lower learning rate (1e-5) as a secondary safeguard. The primary defense against catastrophic forgetting is the ConcatDataset mixed training from Step 6.2.

In [5]:
import torch.optim as optim
from transformers import get_cosine_schedule_with_warmup

# ============================================================
# Hyperparameters
# ============================================================
EPOCHS             = 10
FREEZE_EPOCHS      = 2       # Head-only training before unfreezing backbone
learning_rate      = 1e-5    # Lower than Phase 2's 3e-5
weight_decay       = 0.05
ACCUMULATION_STEPS = 4       # Effective batch size = 8 × 4 = 32
patience           = 3       # Early stopping on val loss
CHECKPOINT_PATH    = "best_vit_deepfake_3.pth"

# Loss function
criterion = nn.BCEWithLogitsLoss()

# ============================================================
# Phase A Optimizer: Head-only (epochs 1–2)
# ============================================================
head_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(head_params, lr=learning_rate, weight_decay=weight_decay)

head_total_steps = (len(train_loader) // ACCUMULATION_STEPS) * FREEZE_EPOCHS
head_warmup      = int(0.10 * head_total_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=head_warmup, num_training_steps=head_total_steps
)

print("Phase 6 Fine-tuning Configuration:")
print(f"  Epochs:             {EPOCHS} (first {FREEZE_EPOCHS} with frozen backbone)")
print(f"  Learning Rate:      {learning_rate}")
print(f"  Weight Decay:       {weight_decay}")
print(f"  Batch Size:         {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"  Accumulation Steps: {ACCUMULATION_STEPS}")
print(f"  Early Stopping:     patience={patience} (watching val loss)")
print(f"  Checkpoint:         {CHECKPOINT_PATH}")
print(f"\nPhase A (Head Warmup — epochs 1-{FREEZE_EPOCHS}):")
print(f"  Optimizer params:   {sum(p.numel() for p in head_params):,}")
print(f"  Scheduler steps:    {head_total_steps} (warmup: {head_warmup})")

Phase 6 Fine-tuning Configuration:
  Epochs:             10 (first 2 with frozen backbone)
  Learning Rate:      1e-05
  Weight Decay:       0.05
  Batch Size:         8 (effective: 32)
  Accumulation Steps: 4
  Early Stopping:     patience=3 (watching val loss)
  Checkpoint:         best_vit_deepfake_3.pth

Phase A (Head Warmup — epochs 1-2):
  Optimizer params:   769
  Scheduler steps:    4738 (warmup: 473)


In [6]:
# ============================================================
# Verification: Quick forward pass on a mixed batch
# ============================================================
print("Extracting a test batch from the mixed training loader...")
test_images, test_labels = next(iter(train_loader))
print(f"Batch shape: images={test_images.shape}, labels={test_labels.shape}")
print(f"Unique labels in batch: {test_labels.unique().tolist()}")

test_images_dev = test_images.to(device)
test_labels_dev = test_labels.to(device).float().unsqueeze(1)

model.train()
with torch.set_grad_enabled(True):
    test_logits = model(test_images_dev)
    test_loss = criterion(test_logits, test_labels_dev)
    print(f"Forward pass ✅  Logits: {test_logits.shape}, Loss: {test_loss.item():.4f}")

Extracting a test batch from the mixed training loader...
Batch shape: images=torch.Size([8, 3, 224, 224]), labels=torch.Size([8])
Unique labels in batch: [0, 1]
Forward pass ✅  Logits: torch.Size([8, 1]), Loss: 1.2144


# Step 6.5: Training & Evaluation Loop

- **Epochs 1–2:** Head-only training (backbone frozen) — calibrates the classifier to the mixed distribution.
- **Epoch 3:** Unfreeze backbone, create new optimizer with all parameters.
- **Epochs 3–10:** Full fine-tuning with early stopping (patience=3, watching val loss).
- Checkpoints saved on best combined validation ROC-AUC → `best_vit_deepfake_3.pth`  (607 min)

In [7]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from tqdm.notebook import tqdm

# ============================================================
# Training & Evaluation Loop
# ============================================================

history = {
    "train_loss": [], "val_loss": [],
    "val_accuracy": [], "val_precision": [],
    "val_recall": [], "val_f1": [], "val_auc": []
}

best_val_auc     = 0.0
best_val_loss    = float('inf')
patience_counter = 0

print(f"Starting Phase 6 fine-tuning on device: {device}")
print(f"Mixed dataset: {len(combined_train):,} train | {len(combined_val):,} val\n")
global_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # ==== BACKBONE UNFREEZE CHECK ====
    if epoch == FREEZE_EPOCHS + 1:
        print(f"\n{'=' * 60}")
        print(f"  🔓 UNFREEZING ViT backbone at epoch {epoch}")
        print(f"{'=' * 60}")
        model.vit.requires_grad_(True)

        # New optimizer with ALL parameters
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        # New scheduler for remaining epochs
        remaining_steps  = (len(train_loader) // ACCUMULATION_STEPS) * (EPOCHS - FREEZE_EPOCHS)
        remaining_warmup = int(0.10 * remaining_steps)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=remaining_warmup, num_training_steps=remaining_steps
        )

        trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable parameters: {trainable_now:,} (all)")
        print(f"  New scheduler: {remaining_steps} steps (warmup: {remaining_warmup})")
        print(f"{'=' * 60}\n")

    # ==== TRAINING PHASE ====
    model.train()
    running_loss      = 0.0
    processed_samples = 0
    phase_label = "HEAD-ONLY" if epoch <= FREEZE_EPOCHS else "FULL"

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    optimizer.zero_grad()
    log_interval = max(1, len(train_loader) // 10)

    for batch_idx, (images, labels) in enumerate(train_bar):
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        logits = model(images)
        loss   = criterion(logits, labels)

        # Gradient accumulation
        loss = loss / ACCUMULATION_STEPS
        loss.backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        running_loss      += loss.item() * ACCUMULATION_STEPS * images.size(0)
        processed_samples += images.size(0)
        train_bar.set_postfix(loss=f"{(loss.item() * ACCUMULATION_STEPS):.4f}")

        # Sub-progress logging every 10%
        if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
            elapsed_batch  = time.time() - epoch_start
            it_per_sec     = (batch_idx + 1) / elapsed_batch
            eta_sec        = (len(train_loader) - batch_idx - 1) / it_per_sec if it_per_sec > 0 else 0
            current_lr     = optimizer.param_groups[0]['lr']
            pct            = int((batch_idx + 1) / len(train_loader) * 100)

            print(f"  -> [{phase_label}] {batch_idx+1}/{len(train_loader)} ({pct}%) | "
                  f"Loss: {running_loss / processed_samples:.4f} | "
                  f"LR: {current_lr:.2e} | ETA: {int(eta_sec // 60)}m {int(eta_sec % 60)}s")

    epoch_train_loss = running_loss / len(combined_train)

    # ==== VALIDATION PHASE ====
    model.eval()
    val_running_loss = 0.0
    all_preds  = []
    all_probs  = []
    all_labels = []

    val_bar = tqdm(combined_val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
    with torch.no_grad():
        for images, labels in val_bar:
            images     = images.to(device)
            labels_dev = labels.to(device).float().unsqueeze(1)

            logits = model(images)
            loss   = criterion(logits, labels_dev)
            val_running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            all_preds.extend(preds.flatten())
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())
            val_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_val_loss = val_running_loss / len(combined_val)

    # Early stopping check
    if epoch_val_loss < best_val_loss:
        best_val_loss    = epoch_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"Warning: Val loss did not improve. Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered! Training halted to protect model weights.")
        break

    # Compute metrics
    accuracy  = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_accuracy"].append(accuracy)
    history["val_precision"].append(precision)
    history["val_recall"].append(recall)
    history["val_f1"].append(f1)
    history["val_auc"].append(auc)

    # Timing
    elapsed_total = time.time() - global_start
    epoch_dur     = time.time() - epoch_start
    remaining_est = epoch_dur * (EPOCHS - epoch)

    print(f"\n--- Epoch {epoch} [{phase_label}] Metrics Summary ---")
    print(f"  Train Loss:       {epoch_train_loss:.4f}")
    print(f"  Val Loss:         {epoch_val_loss:.4f}")
    print(f"  Val Accuracy:     {accuracy:.4f}")
    print(f"  Val Precision:    {precision:.4f}")
    print(f"  Val Recall:       {recall:.4f}")
    print(f"  Val F1-Score:     {f1:.4f}")
    print(f"  Val ROC-AUC:      {auc:.4f}")
    print(f"  Epoch Duration:   {int(epoch_dur // 60)}m {int(epoch_dur % 60)}s")
    print(f"  Elapsed Time:     {int(elapsed_total // 60)}m {int(elapsed_total % 60)}s")
    if EPOCHS - epoch > 0:
        print(f"  Est. Remaining:   {int(remaining_est // 60)}m {int(remaining_est % 60)}s")

    # Checkpoint on best ROC-AUC
    if auc > best_val_auc:
        print(f"  => Val ROC-AUC improved {best_val_auc:.4f} → {auc:.4f}. "
              f"Saving to '{CHECKPOINT_PATH}'...")
        best_val_auc = auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
    print("-" * 40)

total_time = time.time() - global_start
print(f"\n{'=' * 60}")
print(f"  Training Complete!")
print(f"  Total time:       {int(total_time // 3600)}h "
      f"{int((total_time % 3600) // 60)}m {int(total_time % 60)}s")
print(f"  Best Val ROC-AUC: {best_val_auc:.4f}")
print(f"  Checkpoint saved: {CHECKPOINT_PATH}")
print(f"{'=' * 60}")

Starting Phase 6 fine-tuning on device: mps
Mixed dataset: 75,824 train | 29,667 val



Epoch 1/10 [Train]:   0%|          | 0/9478 [00:00<?, ?it/s]

  -> [HEAD-ONLY] 947/9478 (9%) | Loss: 0.9723 | LR: 4.99e-06 | ETA: 24m 53s
  -> [HEAD-ONLY] 1894/9478 (19%) | Loss: 0.9383 | LR: 1.00e-05 | ETA: 23m 56s
  -> [HEAD-ONLY] 2841/9478 (29%) | Loss: 0.9070 | LR: 9.92e-06 | ETA: 22m 6s
  -> [HEAD-ONLY] 3788/9478 (39%) | Loss: 0.8897 | LR: 9.70e-06 | ETA: 19m 48s
  -> [HEAD-ONLY] 4735/9478 (49%) | Loss: 0.8667 | LR: 9.33e-06 | ETA: 17m 7s
  -> [HEAD-ONLY] 5682/9478 (59%) | Loss: 0.8522 | LR: 8.83e-06 | ETA: 14m 7s
  -> [HEAD-ONLY] 6629/9478 (69%) | Loss: 0.8322 | LR: 8.22e-06 | ETA: 10m 53s
  -> [HEAD-ONLY] 7576/9478 (79%) | Loss: 0.8152 | LR: 7.50e-06 | ETA: 7m 26s
  -> [HEAD-ONLY] 8523/9478 (89%) | Loss: 0.8009 | LR: 6.72e-06 | ETA: 3m 48s
  -> [HEAD-ONLY] 9470/9478 (99%) | Loss: 0.7864 | LR: 5.87e-06 | ETA: 0m 1s
  -> [HEAD-ONLY] 9478/9478 (100%) | Loss: 0.7861 | LR: 5.86e-06 | ETA: 0m 0s


Epoch 1/10 [Val]:   0%|          | 0/3709 [00:00<?, ?it/s]


--- Epoch 1 [HEAD-ONLY] Metrics Summary ---
  Train Loss:       0.7861
  Val Loss:         0.8807
  Val Accuracy:     0.6516
  Val Precision:    0.6073
  Val Recall:       0.8694
  Val F1-Score:     0.7151
  Val ROC-AUC:      0.7105
  Epoch Duration:   54m 6s
  Elapsed Time:     54m 6s
  Est. Remaining:   486m 56s
  => Val ROC-AUC improved 0.0000 → 0.7105. Saving to 'best_vit_deepfake_3.pth'...
----------------------------------------


...

Epoch 5/10 [Val]:   0%|          | 0/3709 [00:00<?, ?it/s]


--- Epoch 5 [FULL] Metrics Summary ---
  Train Loss:       0.0402
  Val Loss:         0.1039
  Val Accuracy:     0.9677
  Val Precision:    0.9683
  Val Recall:       0.9675
  Val F1-Score:     0.9679
  Val ROC-AUC:      0.9943
  Epoch Duration:   168m 56s
  Elapsed Time:     593m 13s
  Est. Remaining:   844m 44s
  => Val ROC-AUC improved 0.9923 → 0.9943. Saving to 'best_vit_deepfake_3.pth'...
----------------------------------------


Epoch 6/10 [Train]:   0%|          | 0/9478 [00:00<?, ?it/s]

  -> [FULL] 947/9478 (9%) | Loss: 0.0300 | LR: 7.68e-06 | ETA: 129m 49s


: 

# Step 6.5 (cont.): Per-Dataset Final Evaluation

Evaluate `best_vit_deepfake_3.pth` separately on the DFDC validation set, StyleGAN validation set, and StyleGAN test set.
This verifies both **retention** of face-swap detection and **acquisition** of StyleGAN detection.

In [4]:
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor, ViTModel
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)
from tqdm.notebook import tqdm

# ============================================================
# Self-contained setup (safe to run after kernel restart)
# ============================================================

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

# Paths
DFDC_ROOT       = "processed_faces_v2"
STYLEGAN_ROOT   = "Fake_face_detection_with_Keras/real_vs_fake/real-vs-fake"
CHECKPOINT_PATH = "best_vit_deepfake_3.pth"

# Transforms
model_name = "google/vit-base-patch16-224-in21k"
processor  = ViTImageProcessor.from_pretrained(model_name)
mean, std  = processor.image_mean, processor.image_std
size       = processor.size.get('height', 224)

val_transforms = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Model
class ViTDeepfakeClassifier(nn.Module):
    def __init__(self, vit_model, dropout_rate=0.2):
        super().__init__()
        self.vit = vit_model
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.vit.config.hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

vit_base = ViTModel.from_pretrained(model_name)
model    = ViTDeepfakeClassifier(vit_base, dropout_rate=0.2)

# DataLoaders
BATCH_SIZE  = 8
NUM_WORKERS = 2

dfdc_val       = ImageFolder(root=os.path.join(DFDC_ROOT, "val"),       transform=val_transforms)
stylegan_val   = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "valid"), transform=val_transforms)
stylegan_test  = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "test"),  transform=val_transforms)

dfdc_val_loader      = DataLoader(dfdc_val,      batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
stylegan_val_loader  = DataLoader(stylegan_val,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
stylegan_test_loader = DataLoader(stylegan_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

criterion = nn.BCEWithLogitsLoss()

# ============================================================
# Per-Dataset Evaluation Function
# ============================================================
def evaluate_on_dataset(model, loader, dataset_name, device, criterion):
    """Evaluate the model on a single dataset and return metrics."""
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss    = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Evaluating {dataset_name}"):
            images     = images.to(device)
            labels_dev = labels.to(device).float().unsqueeze(1)

            logits = model(images)
            loss   = criterion(logits, labels_dev)
            total_loss    += loss.item() * images.size(0)
            total_samples += images.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            all_preds.extend(preds.flatten())
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    cm  = confusion_matrix(all_labels, all_preds)

    return {
        'loss': total_loss / total_samples,
        'accuracy': accuracy, 'precision': precision,
        'recall': recall, 'f1': f1, 'auc': auc,
        'confusion_matrix': cm
    }

# ============================================================
# Load Best Checkpoint
# ============================================================
print(f"Loading best checkpoint: {CHECKPOINT_PATH}")
best_state = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(best_state)
model = model.to(device)

# ============================================================
# Evaluate on Each Dataset
# ============================================================
separator = '=' * 60
print(f"\n{separator}")
print(f"  PER-DATASET EVALUATION — {CHECKPOINT_PATH}")
print(separator)

eval_sets = {
    "DFDC Validation":     dfdc_val_loader,
    "StyleGAN Validation": stylegan_val_loader,
    "StyleGAN Test":       stylegan_test_loader,
}

all_results = {}
for name, loader in eval_sets.items():
    print(f"\n--- {name} ---")
    results = evaluate_on_dataset(model, loader, name, device, criterion)
    all_results[name] = results

    cm = results['confusion_matrix']
    print(f"  Loss:      {results['loss']:.4f}")
    print(f"  Accuracy:  {results['accuracy']:.4f}  ({results['accuracy'] * 100:.2f}%)")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall:    {results['recall']:.4f}")
    print(f"  F1-Score:  {results['f1']:.4f}")
    print(f"  ROC-AUC:   {results['auc']:.4f}")
    print(f"  Confusion Matrix: TN={cm[0][0]}  FP={cm[0][1]}  |  FN={cm[1][0]}  TP={cm[1][1]}")

# ============================================================
# Summary Table
# ============================================================
wide_sep = '=' * 70
dash_sep = '-' * 55
print(f"\n\n{wide_sep}")
print("  SUMMARY TABLE — v3 (Mixed Fine-tuned) Performance")
print(wide_sep)
header = f"{'Dataset':<25} {'Accuracy':>10} {'F1':>10} {'ROC-AUC':>10}"
print(header)
print(dash_sep)
for name, r in all_results.items():
    print(f"{name:<25} {r['accuracy']:>10.4f} {r['f1']:>10.4f} {r['auc']:>10.4f}")
print(wide_sep)

# DFDC Retention Check
dfdc_acc = all_results["DFDC Validation"]["accuracy"]
dfdc_auc = all_results["DFDC Validation"]["auc"]
if dfdc_acc >= 0.85:
    print(f"\n✅ DFDC retention check PASSED: {dfdc_acc:.2%} accuracy, {dfdc_auc:.4f} AUC")
else:
    print(f"\n⚠️  DFDC retention WARNING: {dfdc_acc:.2%} accuracy (below 85% threshold)")

sg_acc = all_results["StyleGAN Test"]["accuracy"]
sg_auc = all_results["StyleGAN Test"]["auc"]
if sg_acc >= 0.70:
    print(f"✅ StyleGAN detection PASSED: {sg_acc:.2%} accuracy, {sg_auc:.4f} AUC")
else:
    print(f"⚠️  StyleGAN detection WARNING: {sg_acc:.2%} accuracy (below 70% threshold)")

file_size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
print(f"\nCheckpoint: {CHECKPOINT_PATH} ({file_size_mb:.2f} MB)")


Device: mps


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading best checkpoint: best_vit_deepfake_3.pth

  PER-DATASET EVALUATION — best_vit_deepfake_3.pth

--- DFDC Validation ---


Evaluating DFDC Validation:   0%|          | 0/1209 [00:00<?, ?it/s]

  Loss:      0.2693
  Accuracy:  0.9175  (91.75%)
  Precision: 0.9182
  Recall:    0.9197
  F1-Score:  0.9190
  ROC-AUC:   0.9752
  Confusion Matrix: TN=4345  FP=403  |  FN=395  TP=4524

--- StyleGAN Validation ---


Evaluating StyleGAN Validation:   0%|          | 0/2500 [00:00<?, ?it/s]

  Loss:      0.0239
  Accuracy:  0.9920  (99.20%)
  Precision: 0.9930
  Recall:    0.9910
  F1-Score:  0.9920
  ROC-AUC:   0.9997
  Confusion Matrix: TN=9930  FP=70  |  FN=90  TP=9910

--- StyleGAN Test ---


Evaluating StyleGAN Test:   0%|          | 0/2500 [00:00<?, ?it/s]

  Loss:      0.0245
  Accuracy:  0.9925  (99.25%)
  Precision: 0.9938
  Recall:    0.9911
  F1-Score:  0.9924
  ROC-AUC:   0.9995
  Confusion Matrix: TN=9938  FP=62  |  FN=89  TP=9911


  SUMMARY TABLE — v3 (Mixed Fine-tuned) Performance
Dataset                     Accuracy         F1    ROC-AUC
-------------------------------------------------------
DFDC Validation               0.9175     0.9190     0.9752
StyleGAN Validation           0.9920     0.9920     0.9997
StyleGAN Test                 0.9925     0.9924     0.9995

✅ DFDC retention check PASSED: 91.75% accuracy, 0.9752 AUC
✅ StyleGAN detection PASSED: 99.25% accuracy, 0.9995 AUC

Checkpoint: best_vit_deepfake_3.pth (329.62 MB)


# Step 6.6: Cross-Dataset Evaluation

Run `best_vit_deepfake_3.pth` on the original DFDC `val` set to verify it has not forgotten face-swap detection.
Run `best_vit_deepfake_2.pth` on the StyleGAN `test` set to document the baseline failure rate.
This cross-test confirms whether the fine-tuned model is truly dual-capability.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor, ViTModel
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)
from tqdm.notebook import tqdm

# Self-contained setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

DFDC_ROOT     = "processed_faces_v2"
STYLEGAN_ROOT = "Fake_face_detection_with_Keras/real_vs_fake/real-vs-fake"
model_name    = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(model_name)
mean, std = processor.image_mean, processor.image_std
size      = processor.size.get('height', 224)

val_transforms = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class ViTDeepfakeClassifier(nn.Module):
    def __init__(self, vit_model, dropout_rate=0.2):
        super().__init__()
        self.vit = vit_model
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.vit.config.hidden_size, 1)
    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_output))

BATCH_SIZE  = 8
NUM_WORKERS = 2
criterion   = nn.BCEWithLogitsLoss()

dfdc_val      = ImageFolder(root=os.path.join(DFDC_ROOT, "val"),       transform=val_transforms)
stylegan_test = ImageFolder(root=os.path.join(STYLEGAN_ROOT, "test"),  transform=val_transforms)

dfdc_val_loader      = DataLoader(dfdc_val,      batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
stylegan_test_loader = DataLoader(stylegan_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

def evaluate(model, loader, name, device):
    """Evaluate model on a dataset. Returns metrics dict."""
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=name):
            images = images.to(device)
            labels_dev = labels.to(device).float().unsqueeze(1)
            logits = model(images)
            loss = criterion(logits, labels_dev)
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds.flatten())
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return {'loss': total_loss/total_samples, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'auc': auc, 'cm': cm}

# Evaluate both checkpoints on both datasets
checkpoints = {
    'v2 (DFDC-only)':        'best_vit_deepfake_2.pth',
    'v3 (Mixed fine-tuned)': 'best_vit_deepfake_3.pth',
}
test_sets = {
    'DFDC Val':      dfdc_val_loader,
    'StyleGAN Test': stylegan_test_loader,
}

vit_base = ViTModel.from_pretrained(model_name)
results = {}

for ckpt_name, ckpt_path in checkpoints.items():
    if not os.path.exists(ckpt_path):
        print(f"\n  {ckpt_path} not found, skipping.")
        continue
    file_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f"\n{'=' * 60}")
    print(f"  Loading: {ckpt_name} ({ckpt_path}, {file_mb:.1f} MB)")
    print('=' * 60)
    model = ViTDeepfakeClassifier(vit_base, dropout_rate=0.2)
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model = model.to(device)
    results[ckpt_name] = {}
    for ds_name, loader in test_sets.items():
        tag = f"{ckpt_name} on {ds_name}"
        r = evaluate(model, loader, tag, device)
        results[ckpt_name][ds_name] = r
        cm = r["cm"]
        print(f"\n  {ds_name}:")
        print(f"    Accuracy:  {r['accuracy']:.4f}  ({r['accuracy']*100:.2f}%)")
        print(f"    Precision: {r['precision']:.4f}")
        print(f"    Recall:    {r['recall']:.4f}")
        print(f"    F1-Score:  {r['f1']:.4f}")
        print(f"    ROC-AUC:   {r['auc']:.4f}")
        print(f"    TN={cm[0][0]}  FP={cm[0][1]}  |  FN={cm[1][0]}  TP={cm[1][1]}")

print("\nCross-dataset evaluation complete. Run the next cells for analysis.")


# Step 6.7: Dual-Model Strategy Decision

Based on cross-dataset results from Step 6.6, decide on integration strategy:
- **Option A:** Single unified model if v3 retains DFDC accuracy.
- **Option B:** Ensemble / routing if DFDC performance degrades.

In [ ]:
# Step 6.7: Strategy Decision
ckpt_names = list(results.keys())
assert len(ckpt_names) == 2, "Need both v2 and v3 results from Step 6.6"

r_v2 = results[ckpt_names[0]]
r_v3 = results[ckpt_names[1]]

dfdc_v2_acc = r_v2["DFDC Val"]["accuracy"]
dfdc_v3_acc = r_v3["DFDC Val"]["accuracy"]
dfdc_v2_auc = r_v2["DFDC Val"]["auc"]
dfdc_v3_auc = r_v3["DFDC Val"]["auc"]

sg_v2_acc = r_v2["StyleGAN Test"]["accuracy"]
sg_v3_acc = r_v3["StyleGAN Test"]["accuracy"]
sg_v2_auc = r_v2["StyleGAN Test"]["auc"]
sg_v3_auc = r_v3["StyleGAN Test"]["auc"]

retention_ratio = dfdc_v3_acc / dfdc_v2_acc if dfdc_v2_acc > 0 else 0

print('=' * 60)
print("  STRATEGY DECISION")
print('=' * 60)

print(f"\n  DFDC Retention:")
print(f"    v2 Accuracy: {dfdc_v2_acc:.4f}  |  v3 Accuracy: {dfdc_v3_acc:.4f}")
print(f"    v2 ROC-AUC:  {dfdc_v2_auc:.4f}  |  v3 ROC-AUC:  {dfdc_v3_auc:.4f}")
print(f"    Retention ratio: {retention_ratio:.2%}")

print(f"\n  StyleGAN Capability:")
print(f"    v2 Accuracy: {sg_v2_acc:.4f}  |  v3 Accuracy: {sg_v3_acc:.4f}")
print(f"    v2 ROC-AUC:  {sg_v2_auc:.4f}  |  v3 ROC-AUC:  {sg_v3_auc:.4f}")

print('\n' + '-' * 50)

if retention_ratio >= 0.90 and sg_v3_acc > 0.70:
    strategy = "A"
    print("\n  RECOMMENDATION: Option A -- Single Unified Model")
    print("  Deploy only best_vit_deepfake_3.pth in app.py.")
    print(f"  DFDC retention: {retention_ratio:.1%} (>= 90% threshold)")
    print(f"  StyleGAN detection: {sg_v3_acc:.2%} (>= 70% threshold)")
elif retention_ratio >= 0.90:
    strategy = "A-weak"
    print("\n  RECOMMENDATION: Option A (weak) -- Single Model")
    print("  DFDC retained but StyleGAN detection is below 70%.")
    print("  Model may need more training or data.")
else:
    strategy = "B"
    print("\n  RECOMMENDATION: Option B -- Ensemble / Routing")
    print("  Run both models in parallel in app.py and aggregate scores.")
    print(f"  DFDC retention: {retention_ratio:.1%} (below 90% threshold)")

print('\n' + '=' * 60)
print(f"  Strategy chosen: Option {strategy}")
print('=' * 60)


# Step 6.9: Side-by-Side Comparison Table

Formatted comparison of `best_vit_deepfake_2.pth` vs `best_vit_deepfake_3.pth` across both datasets.

In [ ]:
# Step 6.9: Side-by-Side Comparison Table
ckpt_names = list(results.keys())
r_v2 = results[ckpt_names[0]]
r_v3 = results[ckpt_names[1]]

wide_sep = '=' * 78
dash_sep = '-' * 70

print(f"\n{wide_sep}")
print("  SIDE-BY-SIDE COMPARISON: v2 (DFDC-only) vs v3 (Mixed fine-tuned)")
print(wide_sep)

for ds_name in ['DFDC Val', 'StyleGAN Test']:
    v2 = r_v2[ds_name]
    v3 = r_v3[ds_name]
    print(f"\n  {ds_name}")
    print(f"  {dash_sep}")
    print(f"  {'Metric':<14} {'v2':>10} {'v3':>10} {'Delta':>10} {'':<6}")
    print(f"  {dash_sep}")

    for metric, label in [('accuracy','Accuracy'), ('precision','Precision'),
                           ('recall','Recall'), ('f1','F1-Score'), ('auc','ROC-AUC')]:
        v2_val = v2[metric]
        v3_val = v3[metric]
        delta = v3_val - v2_val
        arrow = 'UP' if delta > 0 else 'DN' if delta < 0 else '=='
        status = '+' if delta > 0 else '-' if delta < 0 else '='
        print(f"  {label:<14} {v2_val:>10.4f} {v3_val:>10.4f} {arrow}{abs(delta):>8.4f} [{status}]")

print(f"\n{wide_sep}")

# Final verdict count
v3_wins_dfdc = 0
v3_wins_sg = 0
for m in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    if r_v3["DFDC Val"][m] >= r_v2["DFDC Val"][m]:
        v3_wins_dfdc += 1
    if r_v3["StyleGAN Test"][m] >= r_v2["StyleGAN Test"][m]:
        v3_wins_sg += 1

print(f"\nv3 wins on DFDC Val:      {v3_wins_dfdc}/5 metrics")
print(f"v3 wins on StyleGAN Test: {v3_wins_sg}/5 metrics")

if v3_wins_dfdc >= 3 and v3_wins_sg >= 3:
    print("\nVerdict: v3 is the superior model across both domains.")
elif v3_wins_sg >= 4:
    print("\nVerdict: v3 dominates StyleGAN detection with acceptable DFDC trade-off.")
else:
    print("\nVerdict: Mixed results -- review the numbers above to decide.")
